# Lab 2 · Tools และ agent loop

ใน Lab 1 LLM อ่านเอกสารแล้วตอบ แต่ลงมือทำอะไรไม่ได้ Lab นี้ให้ LLM มี **tools**: function ที่เปิด PDF, คำนวณยอด, ค้นใบสั่งซื้อในฐานข้อมูล และบันทึกผล แล้วให้มันเลือกเองว่าจะเรียก tool ไหนเมื่อไร

**งาน:** ตรวจใบแจ้งหนี้ PDF 8 ฉบับเทียบกับใบสั่งซื้อ (purchase order, PO) ในฐานข้อมูล แล้วบันทึกว่าอนุมัติหรือปฏิเสธ พร้อมเหตุผล

| ส่วน | สิ่งที่ทำ | เวลา |
|---|---|---|
| 1 | Tool คือ function ที่มีคำอธิบาย | 10 นาที |
| 2 | เรียก tool หนึ่งรอบด้วยมือ | 10 นาที |
| 3 | เขียน agent loop เอง | 15 นาที |
| 4 | ตรวจครบ 8 ฉบับและวัดผล | 10 นาที |
| 5 | การออกแบบ tool และการรับมือ error | 10 นาที |
| 6 | `create_agent` และเทียบกับ workflow | 10 นาที |
| 7 | Stretch: ให้บริการ tools ผ่าน MCP | ถ้ามีเวลา |

มองหา `# TODO` ใน code ทุก cell รันได้แม้ยังไม่แก้ ก่อนเริ่มให้บันทึกไฟล์นี้เป็น `lab2_tool_agent.mine.ipynb`

## 1. Tool คือ function ที่มีคำอธิบาย (10 นาที)

Tools ของ lab นี้อยู่ใน `agentlab/invoices.py` แต่ละตัวเป็น function ธรรมดาที่ครอบด้วย `@tool` เราเรียกมันตรงๆ ได้เหมือน function ทั่วไป ผ่าน `.invoke()`

In [ ]:
import json
import time

import pandas as pd
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

from agentlab import get_llm, release_model
from agentlab.invoices import (INVOICE_DIR, SYSTEM_PROMPT, TOOLS, check_invoice_total, compare_with_po,
                               create_database, evaluate_checks, read_invoice, record_result)

create_database(reset=True)  # a fresh database: purchase orders, and no results yet
llm = get_llm()

print(read_invoice.invoke({"filename": "inv_04.pdf"}))
print()
print(check_invoice_total.invoke({"filename": "inv_04.pdf"}))

Model มองไม่เห็น code ของ tool มันเห็นเฉพาะ **ชื่อ คำอธิบาย และ argument** ที่ LangChain สร้างจาก function cell ด้านล่างแสดงสิ่งที่ model ได้รับจริง

In [ ]:
for t in TOOLS:
    spec = convert_to_openai_tool(t)["function"]
    print(f"{spec['name']}({', '.join(spec['parameters']['properties'])})")
    print(f"    {spec['description']}\n")

print(json.dumps(convert_to_openai_tool(record_result), indent=2))

**งานของคุณ:** เขียน tool `list_invoices` ที่คืนรายชื่อไฟล์ใบแจ้งหนี้ทั้งหมดใน `INVOICE_DIR` เป็น JSON list เขียน docstring ที่บอก model ว่า tool นี้ทำอะไรและควรใช้เมื่อไร แล้วดูว่า schema ที่ได้เป็นอย่างไร

In [ ]:
@tool
def list_invoices() -> str:
    """TODO: one or two sentences for the model: what this returns and when to use it."""
    # TODO: return json.dumps of the sorted file names (p.name) of INVOICE_DIR.glob("*.pdf")
    return "[]"


print(list_invoices.invoke({}))
print(convert_to_openai_tool(list_invoices)["function"])

## 2. เรียก tool หนึ่งรอบด้วยมือ (10 นาที)

`bind_tools()` แนบคำอธิบายของ tools ไปกับทุก request เมื่อ model อยากใช้ tool มันจะ **ไม่ตอบเป็นข้อความ** แต่ตอบเป็น `tool_calls`: ชื่อ tool และ argument ที่ต้องการ

In [ ]:
DEMO_SYSTEM = "You answer questions about invoices using the tools. Never do arithmetic yourself."
model = llm.bind_tools(TOOLS)
messages = [SystemMessage(DEMO_SYSTEM), HumanMessage("Is the total on inv_04.pdf correct?")]

reply = model.invoke(messages)
print("text:      ", repr(reply.text))
print("tool_calls:", reply.tool_calls)

Model แค่ **ขอ** ให้เรียก tool ยังไม่มีอะไรถูกรัน ตอนนี้เราเป็น harness: รัน tool เอง แล้วส่งผลกลับเป็น `ToolMessage` ที่อ้าง `tool_call_id` เดียวกัน จากนั้นถาม model อีกครั้ง

In [ ]:
tools_by_name = {t.name: t for t in TOOLS}

call = reply.tool_calls[0]
result = tools_by_name[call["name"]].invoke(call["args"])
print("tool result:", result)

messages += [reply, ToolMessage(content=result, tool_call_id=call["id"])]
final = model.invoke(messages)
print("\nfinal answer:", final.text)
print("\nmessages so far:", [type(m).__name__ for m in messages + [final]])

## 3. เขียน agent loop เอง (15 นาที)

ส่วนที่ 2 คือหนึ่งรอบ งานจริงต้องหลายรอบ: model เรียก tool, เราส่งผลกลับ, model เรียก tool ถัดไป จนกว่ามันจะตอบโดยไม่เรียก tool อีก นั่นคือ agent loop

**งานของคุณ:** เติม `TODO 1` และ `TODO 2` ใน `run_agent` ส่วน `run_tool` แยกไว้ต่างหาก เพราะเราจะทำให้มันปลอดภัยขึ้นในส่วนที่ 5

In [ ]:
def run_tool(call, tools):
    """Run one tool call the model asked for and return the result as text."""
    return {t.name: t for t in tools}[call["name"]].invoke(call["args"])


def run_agent(task, tools=TOOLS, system=SYSTEM_PROMPT, max_steps=8):
    """Call the model, run the tools it asks for, and repeat until it stops asking. Returns (answer, messages)."""
    model = llm.bind_tools(tools)
    messages = [SystemMessage(system), HumanMessage(task)]
    for step in range(max_steps):
        reply = model.invoke(messages)
        messages.append(reply)
        # TODO 1: if reply.tool_calls is empty, the agent has finished: return reply.text, messages
        # TODO 2: otherwise, for each call in reply.tool_calls, append
        #         ToolMessage(content=run_tool(call, tools), tool_call_id=call["id"])
    return "(stopped: max_steps reached)", messages


def show_trace(messages):
    """Print what happened at each step of a run."""
    for m in messages:
        if isinstance(m, HumanMessage):
            print(f"TASK   {m.content}")
        elif getattr(m, "tool_calls", None):
            for c in m.tool_calls:
                print(f"CALL   {c['name']}({json.dumps(c['args'], ensure_ascii=False)})")
        elif isinstance(m, ToolMessage):
            print(f"RESULT {m.text[:110]}")
        elif m.type == "ai":
            print(f"ANSWER {m.text}")


answer, messages = run_agent("Check invoice inv_03.pdf.")
show_trace(messages)

## 4. ตรวจครบ 8 ฉบับและวัดผล (10 นาที)

รัน agent กับใบแจ้งหนี้ทุกฉบับ แล้วเทียบสิ่งที่ `record_result` บันทึกไว้กับเฉลยใน `data/invoices_labels.csv` การปฏิเสธจะนับว่าถูกก็ต่อเมื่อเหตุผลระบุ check ที่ล้มเหลวจริง ใช้เวลาราว 1–3 นาที

In [ ]:
def run_all(runner):
    """Check every invoice with runner(task) -> (answer, messages). Returns per-invoice steps and time."""
    create_database(reset=True)
    rows = []
    for path in sorted(INVOICE_DIR.glob("*.pdf")):
        start = time.time()
        answer, messages = runner(f"Check invoice {path.name}.")
        calls = [c["name"] for m in messages for c in (getattr(m, "tool_calls", None) or [])]
        rows.append({"file": path.name, "tool_calls": len(calls), "llm_calls": sum(m.type == "ai" for m in messages),
                     "seconds": round(time.time() - start, 1), "answer": answer[:60]})
    release_model()
    return pd.DataFrame(rows).set_index("file")


runs = run_all(run_agent)
results = evaluate_checks().join(runs)
print(f"correct: {results['correct'].sum()} / {len(results)}   LLM calls: {results['llm_calls'].sum()}   seconds: {results['seconds'].sum():.0f}")
results[["expected_verdict", "issue", "verdict", "reason", "correct", "tool_calls"]]

## 5. การออกแบบ tool และการรับมือ error (10 นาที)

### 5.1 Tool ที่ออกแบบไม่ดี
Tools ใน `agentlab` ผ่านการออกแบบมาแล้ว ตัวด้านล่างทำงานเดียวกับ `compare_with_po` แต่ชื่อกำกวม คำอธิบายมีคำเดียว และให้ model **คัดลอกเลขที่ PO เอง** รันกับ 3 ฉบับแล้วดู trace ว่า model ส่ง argument อะไรมา

In [ ]:
from agentlab.invoices import query


@tool
def compare(po: str) -> str:
    """Compare."""
    rows = query("SELECT * FROM po_lines WHERE po_number = ?", (po,))
    return json.dumps(rows) if rows else "[]"


bad_tools = [read_invoice, check_invoice_total, compare, record_result]
bad_system = SYSTEM_PROMPT.replace("compare_with_po", "compare")
create_database(reset=True)
for name in ["inv_03.pdf", "inv_06.pdf", "inv_08.pdf"]:
    print(f"===== {name}")
    answer, messages = run_agent(f"Check invoice {name}.", tools=bad_tools, system=bad_system)
    show_trace(messages[2:])
release_model()
evaluate_checks().loc[["inv_03.pdf", "inv_06.pdf", "inv_08.pdf"], ["expected_verdict", "issue", "verdict", "reason", "correct"]]

สังเกต argument `po` ที่ model ส่งมา: มันเปิดใบแจ้งหนี้อ่านหรือเปล่า หรือแต่งเลขขึ้นเอง และ `inv_08.pdf` ซึ่งไม่มีเลขที่ PO เลยถูกตัดสินอย่างไร ความผิดพลาดแบบนี้ไม่มี error ให้เห็น ระบบดูเหมือนทำงานปกติ

ทางแก้ที่ได้ผลที่สุดอยู่ที่การออกแบบ tool ไม่ใช่การแก้ prompt: `compare_with_po` ใน `agentlab` หาเลขที่ PO จากใบแจ้งหนี้เอง model ส่งแค่ชื่อไฟล์ จึงไม่มีโอกาสแต่งเลข

### 5.2 เมื่อ tool เกิด error
Tool ที่เราเขียนเองอาจ raise exception ได้ เช่น เมื่อ model ส่งชื่อไฟล์ที่ไม่มีอยู่ หรือเรียกชื่อ tool ที่ไม่มีจริง ตอนนี้ `run_tool` ปล่อย exception ออกไป ทำให้ทั้ง loop พัง

**งานของคุณ:** แก้ `run_tool` ให้จับ exception แล้วคืนข้อความ error เป็น JSON ให้ model แทน และตอบอย่างสุภาพเมื่อ model เรียก tool ที่ไม่มีอยู่ จากนั้นรัน cell ซึ่งใช้ tool ที่ raise exception เมื่อไม่พบไฟล์

In [ ]:
def run_tool(call, tools):
    """Run one tool call the model asked for and return the result as text."""
    by_name = {t.name: t for t in tools}
    # TODO: if call["name"] is not in by_name, return a JSON error naming the tools that do exist
    # TODO: wrap the call in try/except Exception and return json.dumps({"error": str(e)}) on failure
    return by_name[call["name"]].invoke(call["args"])


@tool
def strict_total(filename: str) -> str:
    """Recalculate an invoice total. Raises an error if the file does not exist."""
    path = INVOICE_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"{filename} not found; available files are inv_01.pdf to inv_08.pdf")
    return check_invoice_total.invoke({"filename": filename})


try:
    answer, messages = run_agent("Is the total on invoice_9.pdf correct?", tools=[strict_total], system=DEMO_SYSTEM)
    show_trace(messages)
except Exception as error:
    print(f"The loop crashed: {type(error).__name__}: {error}")

## 6. `create_agent` และเทียบกับ workflow (10 นาที)

### 6.1 `create_agent`
LangChain มี agent loop สำเร็จรูปชื่อ `create_agent` ซึ่งทำสิ่งเดียวกับ `run_agent` ของคุณ พร้อมความสามารถที่เราจะใช้ในโมดูลถัดๆ ไป เช่น memory และการหยุดรอคนอนุมัติ ข้างในมันคือ graph ของ LangGraph ซึ่งเราจะเรียนใน Module 3

In [ ]:
from langchain.agents import create_agent

agent = create_agent(llm, TOOLS, system_prompt=SYSTEM_PROMPT)
out = agent.invoke({"messages": [HumanMessage("Check invoice inv_07.pdf.")]})
show_trace(out["messages"])


def run_with_create_agent(task):
    out = agent.invoke({"messages": [HumanMessage(task)]})
    return out["messages"][-1].text, out["messages"]


agent_runs = run_all(run_with_create_agent)
agent_results = evaluate_checks().join(agent_runs)
print(f"\ncreate_agent: correct {agent_results['correct'].sum()} / 8, LLM calls {agent_results['llm_calls'].sum()}, {agent_results['seconds'].sum():.0f}s")

### 6.2 งานนี้ต้องใช้ agent จริงหรือ
ใบแจ้งหนี้ทุกฉบับผ่านขั้นตอนเดียวกันตามลำดับเดิม ตาม Module 1 งานที่บอกลำดับขั้นล่วงหน้าได้ควรเป็น **workflow** function ด้านล่างเรียก tools ชุดเดิมตามลำดับที่เขียนตายตัว โดยไม่เรียก LLM เลย

In [ ]:
def check_with_workflow(filename):
    """The same checks in a fixed order, decided by rules: no LLM."""
    total = json.loads(check_invoice_total.invoke({"filename": filename}))
    po = json.loads(compare_with_po.invoke({"filename": filename}))
    if "error" in total or not total["total_ok"]:
        verdict, reason = "reject", "total"
    elif "error" in po:
        verdict, reason = "reject", "no_po" if "no PO number" in po["error"] else "po_not_found"
    elif not po["matches_po"]:
        verdict, reason = "reject", "unit_price" if "unit price" in " ".join(po["problems"]) else "quantity"
    else:
        verdict, reason = "approve", "all checks passed"
    record_result.invoke({"filename": filename, "verdict": verdict, "reason": reason})


create_database(reset=True)
start = time.time()
for path in sorted(INVOICE_DIR.glob("*.pdf")):
    check_with_workflow(path.name)
workflow_seconds = time.time() - start

pd.DataFrame({
    "correct (of 8)": [results["correct"].sum(), agent_results["correct"].sum(), evaluate_checks()["correct"].sum()],
    "LLM calls": [results["llm_calls"].sum(), agent_results["llm_calls"].sum(), 0],
    "seconds": [round(results["seconds"].sum()), round(agent_results["seconds"].sum()), round(workflow_seconds, 1)],
}, index=["your run_agent", "create_agent", "fixed workflow"])

Workflow ได้ผลเท่ากันหรือดีกว่า โดยไม่เรียก LLM เลยและเร็วกว่ามาก ข้อนี้ไม่ได้แปลว่า agent ไม่มีประโยชน์ แต่แปลว่า **งานนี้** ไม่ต้องการ agent เราใช้งานนี้เพื่อเรียนกลไกของ tool และ loop เพราะเห็นผลชัดและวัดได้

Agent คุ้มค่าเมื่อขั้นตอนเปลี่ยนตามสิ่งที่พบ เช่น ใบแจ้งหนี้ที่อ้างใบสั่งซื้อหลายใบ เอกสารที่ต้องเปิดใบรับของหรือใบลดหนี้ประกอบ หรือคำถามจากผู้ใช้ที่บอกล่วงหน้าไม่ได้ว่าต้องใช้ tool ไหน

## 7. Stretch: ให้บริการ tools ผ่าน MCP

ตอนนี้ tools ใช้ได้เฉพาะใน Python ของเรา **MCP (Model Context Protocol)** เป็นมาตรฐานที่ให้ application ใดก็ได้ค้นหาและเรียก tools ของเราได้ ไฟล์ `mcp_server.py` ในโฟลเดอร์นี้ให้บริการ tools ทั้ง 4 ตัวผ่าน MCP cell ด้านล่างเปิด server นั้นเป็น process แยก ดึงรายการ tools ผ่าน MCP แล้วส่งให้ agent

คำถาม: เทียบ schema ของ `compare_with_po` ที่ได้ผ่าน MCP กับที่ได้ในส่วนที่ 1 อะไรหายไป และมีผลต่อ model อย่างไร

In [ ]:
import sys

from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "invoices": {"command": sys.executable, "args": [str(INVOICE_DIR.parent.parent / "lab-02-tool-use" / "mcp_server.py")], "transport": "stdio"},
})
mcp_tools = await client.get_tools()
for t in mcp_tools:
    print(t.name, "->", t.args_schema)

mcp_agent = create_agent(llm, mcp_tools, system_prompt=SYSTEM_PROMPT)
create_database(reset=True)
out = await mcp_agent.ainvoke({"messages": [HumanMessage("Check invoice inv_08.pdf.")]})
release_model()
show_trace(out["messages"])

## สรุปและสิ่งที่ต้องส่ง

ตอบคำถาม 4 ข้อนี้ ข้อละ 2–3 ประโยค แล้วส่งพร้อมภาพตารางเปรียบเทียบจากส่วนที่ 6.2

1. ในส่วนที่ 2 ใครเป็นผู้รัน `check_invoice_total` จริง model หรือ code ของคุณ และข้อนี้สำคัญต่อความปลอดภัยอย่างไร
2. ในส่วนที่ 5.1 tool `compare` ทำให้ model ผิดพลาดแบบไหน ทำไมการแก้ที่ตัว tool จึงได้ผลกว่าการเขียน prompt ให้ละเอียดขึ้น
3. ทำไม tool ควรคืนข้อความ error ให้ model แทนที่จะ raise exception
4. จากตารางในส่วนที่ 6.2 งานแบบไหนในที่ทำงานของคุณที่ควรใช้ agent และงานแบบไหนที่ workflow ดีกว่า

**จาก notebook สู่โปรแกรม:** เปิด `invoice_agent.py` ในโฟลเดอร์เดียวกัน นั่นคือ agent ตัวนี้ที่เขียนแบบนำไปรันจริง ดูวิธีรันใน `README.md` และคำอธิบายทีละส่วนใน `invoice_agent.th.md`

**Project milestone 2:** ระบุ tools 3–5 ตัวที่ agent ของกลุ่มต้องใช้ เขียนชื่อ คำอธิบาย และ argument ของแต่ละตัวตามหลักในส่วนที่ 5 และบอกว่าตัวไหนมี side effect